In [ ]:
print("Hello! I'm working!")

hi


In [2]:
import pandas as pd


df = pd.read_csv('final_df.csv')

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
from tabfm import TabFMRegressor
from tabfm import tabfm_v1_0_0_jax as tabfm_v1_0_0

def clean_col(c):
    return re.sub(r'[\[\]<>]', '', c)

race_cols = [c for c in df.columns if c.startswith('race_')]
df['race'] = df[race_cols].idxmax(axis=1).str.replace('race_', '', regex=False)

sex_cols = [c for c in df.columns if c.startswith('sex_')]
df['sex'] = df[sex_cols].idxmax(axis=1).str.replace('sex_', '', regex=False)

target_col = 'import_insulin, Insulin [Units/volume] in Serum o'

exclude = {
    'participant_id', 'study_group', 'split',
    target_col,
    'import_glucose, Glucose [Mass/volume] in Serum or',
    'max_glucose',
    'min_glucose',
    'mean_glucose',
    'std_glucose',
    'cv_glucose'
}

feature_cols = [
    c for c in df.columns
    if c not in exclude
    and not c.startswith('race_')
    and not c.startswith('sex_')
]
print('Features:', feature_cols)

tabfm_model = tabfm_v1_0_0.load(model_type='regression')

N_FOLDS = 5
RANDOM_STATE = 42
divider = '=' * 50

forward_results = []
reverse_results = []
prediction_rows = []

for group in df['study_group'].dropna().unique():
    print('')
    print(divider)
    print('Study Group:', group)
    print(divider)

    group_df = df[df['study_group'] == group].copy()
    group_df = group_df.replace([np.inf, -np.inf], np.nan)
    group_df = group_df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

    if len(group_df) < 20:
        print('Skipping - only', len(group_df), 'rows after dropping NAs (too few to model)')
        continue

    X = group_df[feature_cols].copy()
    X['sex'] = X['sex'].astype(str)
    X['race'] = X['race'].astype(str)
    y = group_df[target_col].values
    participant_ids = group_df['participant_id'].values

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_indices = [test_idx for _, test_idx in kf.split(X)]

    # ---------- FORWARD: train on 4 folds, predict held-out fold ----------
    print('')
    print('--- Forward CV (train on other 4 folds -> predict held-out fold) ---')
    for fold_i in range(N_FOLDS):
        test_idx = fold_indices[fold_i]
        train_idx = np.concatenate([fold_indices[j] for j in range(N_FOLDS) if j != fold_i])

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        reg = TabFMRegressor(model=tabfm_model)
        reg.fit(X_train, y_train)
        preds = reg.predict(X_test)
        train_preds = reg.predict(X_train)

        r2 = r2_score(y_test, preds)
        mse = mean_squared_error(y_test, preds)
        train_r2 = r2_score(y_train, train_preds)
        train_mse = mean_squared_error(y_train, train_preds)

        print(f'  Fold {fold_i}: test R2={r2:.4f}  test MSE={mse:.4f}  train R2={train_r2:.4f}  train MSE={train_mse:.4f}')

        forward_results.append({
            'study_group': group, 'fold': fold_i,
            'n_train': len(train_idx), 'n_test': len(test_idx),
            'r2': r2, 'mse': mse, 'train_r2': train_r2, 'train_mse': train_mse
        })

        prediction_rows.append(pd.DataFrame({
            'participant_id': participant_ids[test_idx],
            'study_group': group,
            'fold': fold_i,
            'direction': 'forward',
            'context_fold': 'other_4',
            'actual_insulin': y_test,
            'pred_insulin_tabfm': preds,
        }))

    # ---------- REVERSE: single fold as context -> predict each other fold individually ----------
    print('')
    print('--- Reverse (single fold as context -> predict each other fold individually) ---')
    for context_fold in range(N_FOLDS):
        ctx_idx = fold_indices[context_fold]
        X_ctx, y_ctx = X.iloc[ctx_idx], y[ctx_idx]

        reg = TabFMRegressor(model=tabfm_model)
        reg.fit(X_ctx, y_ctx)

        for query_fold in range(N_FOLDS):
            if query_fold == context_fold:
                continue
            q_idx = fold_indices[query_fold]
            X_q, y_q = X.iloc[q_idx], y[q_idx]

            preds = reg.predict(X_q)
            r2 = r2_score(y_q, preds)
            mse = mean_squared_error(y_q, preds)

            print(f'  Context fold {context_fold} -> Query fold {query_fold}: R2={r2:.4f}  MSE={mse:.4f}')

            reverse_results.append({
                'study_group': group, 'context_fold': context_fold, 'query_fold': query_fold,
                'n_context': len(ctx_idx), 'n_query': len(q_idx),
                'r2': r2, 'mse': mse
            })

            prediction_rows.append(pd.DataFrame({
                'participant_id': participant_ids[q_idx],
                'study_group': group,
                'fold': query_fold,
                'direction': 'reverse',
                'context_fold': context_fold,
                'actual_insulin': y_q,
                'pred_insulin_tabfm': preds,
            }))

print('')
print(divider)
print('Summary - Forward CV (5 folds/group)')
print(divider)
forward_df = pd.DataFrame(forward_results)
print(forward_df)
print('')
print('Forward CV mean R2 by group:')
print(forward_df.groupby('study_group')['r2'].agg(['mean', 'std']))

print('')
print(divider)
print('Summary - Reverse (single-fold-context, 4 query folds/context)')
print(divider)
reverse_df = pd.DataFrame(reverse_results)
print(reverse_df)
print('')
print('Reverse mean R2 by group:')
print(reverse_df.groupby('study_group')['r2'].agg(['mean', 'std']))

prediction_tabfm = pd.concat(prediction_rows, ignore_index=True)
print('')
print('Prediction dataframe:')
print(prediction_tabfm.head())

Features: ['age', 'pulse_vsorres, Heart Rate (bpm)', 'bmi_vsorres, BMI', 'import_hba1c, Hemoglobin A1c/Hemoglobin.total in ', 'import_albumin, Albumin [Mass/volume] in Serum or', 'import_a_g_ratio, Albumin/Globulin ratio', 'import_globulin_total, Globulin [Mass/volume] in ', 'import_protein_total, Protein [Mass/volume] in Se', 'import_c_peptide, C peptide [Mass/volume] in Seru', 'import_calcium, Calcium [Mass/volume] in Serum or', 'import_chloride, Chloride [Moles/volume] in Serum', 'import_sodium, Sodium [Moles/volume] in Serum or ', 'import_potassium, Potassium [Moles/volume] in Ser', 'import_carbon_dioxide_total, Carbon dioxide, tota', 'import_creatinine, Creatinine [Mass/volume] in Se', 'import_bun, Urea nitrogen [Mass/volume] in Serum ', 'import_buncreatinineratio, BUN/Creatinine ratio', 'import_urine_albumin, Albumin [Mass/volume] in Ur', 'import_urine_creatinine, Creatinine [Mass/volume]', 'import_alkaline_phosphatase, Alkaline phosphatase', 'import_alt_got, Alanine aminotransfe

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

groups = prediction_tabfm['study_group'].unique()

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for ax, group in zip(axes.flat, groups):
    group_df = prediction_tabfm[prediction_tabfm['study_group'] == group]
    actual = group_df['actual_insulin']
    preds = group_df['pred_insulin_tabfm']
    r, p = stats.spearmanr(actual, preds)

    ax.scatter(actual, preds, alpha=0.4, s=15, color='steelblue')
    lims = [min(actual.min(), preds.min()), max(actual.max(), preds.max())]
    ax.plot(lims, lims, 'k--', linewidth=1)

    p_str = f'{p:.2e}' if p < 0.001 else f'{p:.3f}'
    ax.text(0.05, 0.93, f'Spearman r = {r:.3f}\np = {p_str}',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax.set_xlabel('Actual Insulin')
    ax.set_ylabel('Predicted Insulin')
    ax.set_title(group)

for ax in axes.flat[len(groups):]:
    ax.axis('off')

fig.suptitle('Actual vs Predicted Insulin (TabFM)', fontsize=14)
plt.tight_layout()
plt.show()
